In [44]:
from rdflib import Graph, URIRef, Namespace
from rdflib.plugins.stores.sparqlstore import SPARQLStore
from tqdm import tqdm
import pandas as pd

In [45]:
#rdflib_dbpedia_graph = Graph("SPARQLStore")
#rdflib_dbpedia_graph.open("http://localhost:9999/bigdata/sparql")

graph = Graph()
graph.parse("D:\FTM\Full-Triple-Matcher\dataset\entity-matching\DW-NB\kg1.ttl")

Failed to convert Literal lexical form to value. Datatype=http://www.w3.org/2001/XMLSchema#gYear, Converter=<function parse_date at 0x000001E852CFF0D0>
Traceback (most recent call last):
  File "C:\Users\Ch.Raghava\miniconda3\envs\full-triple-matcher\lib\site-packages\rdflib\term.py", line 2119, in _castLexicalToPython
    return conv_func(lexical)  # type: ignore[arg-type]
  File "C:\Users\Ch.Raghava\miniconda3\envs\full-triple-matcher\lib\site-packages\isodate\isodates.py", line 203, in parse_date
    raise ISO8601Error('Unrecognised ISO 8601 date format: %r' % datestring)
isodate.isoerror.ISO8601Error: Unrecognised ISO 8601 date format: '-0609'
Failed to convert Literal lexical form to value. Datatype=http://www.w3.org/2001/XMLSchema#gYear, Converter=<function parse_date at 0x000001E852CFF0D0>
Traceback (most recent call last):
  File "C:\Users\Ch.Raghava\miniconda3\envs\full-triple-matcher\lib\site-packages\rdflib\term.py", line 2119, in _castLexicalToPython
    return conv_func(le

<Graph identifier=Nd68a5a5801a84fb0ade79953c3889866 (<class 'rdflib.graph.Graph'>)>

In [46]:
query = """
SELECT DISTINCT ?p
WHERE {
  ?s ?p ?o.
}"""
pred_res_list = graph.query(query)

In [47]:
count_total_query = """
SELECT (COUNT(*) AS ?totalCount)
WHERE {
    ?s <:pred> ?o
}
"""

count_total_obj_query = """
SELECT (COUNT(DISTINCT ?o) AS ?totalCount)
WHERE {
    ?s <:pred> ?o
}
"""

inv_func_dict = dict()
for pred_res in tqdm(pred_res_list):
    query_pred = pred_res[0]

    try:
        replaced_count_total_query = count_total_query.replace(':pred', str(query_pred))
        count_res = graph.query(replaced_count_total_query)
        total_count = 0
        for res in count_res:
            total_count = res.totalCount

        replaced_count_total_obj_query = count_total_obj_query.replace(':pred', str(query_pred))
        count_res = graph.query(replaced_count_total_obj_query)
        total_obj = 0
        for res in count_res:
            total_obj = res.totalCount
        
        inv_func_dict[str(query_pred)] = float(total_obj)/float(total_count)
    except Exception as e:
        print(e)
        print("exception raised: " + str(query_pred))

100%|██████████████████████████████████████████| 545/545 [00:21<00:00, 25.21it/s]


In [48]:
inv_func_list = list()
for key, value in inv_func_dict.items():
    inv_func_list.append({
        'predicate': key,
        'inverse_functionality': value
    })

In [49]:
inv_func_df = pd.DataFrame(inv_func_list)

In [50]:
inv_func_df

,predicate,inverse_functionality
0,http://dbpedia.org/ontology/guest,0.551282
1,http://www.wikidata.org/entity/P345,0.996910
2,http://www.wikidata.org/entity/P161,0.338014
3,http://www.wikidata.org/entity/P175,0.195203
4,http://www.wikidata.org/entity/P577,0.607109
...,...,...
540,http://www.wikidata.org/entity/P38,1.000000
541,http://dbpedia.org/ontology/governmentCountry,1.000000
542,http://dbpedia.org/ontology/feastDay,1.000000
543,http://dbpedia.org/ontology/openingYear,1.000000


In [51]:
inv_func_df[inv_func_df['inverse_functionality'] > 0.5]

,predicate,inverse_functionality
0,http://dbpedia.org/ontology/guest,0.551282
1,http://www.wikidata.org/entity/P345,0.996910
4,http://www.wikidata.org/entity/P577,0.607109
7,http://www.w3.org/2000/01/rdf-schema#label,0.961267
9,http://www.wikidata.org/entity/P569,0.577504
...,...,...
540,http://www.wikidata.org/entity/P38,1.000000
541,http://dbpedia.org/ontology/governmentCountry,1.000000
542,http://dbpedia.org/ontology/feastDay,1.000000
543,http://dbpedia.org/ontology/openingYear,1.000000


In [52]:
query = """
SELECT DISTINCT ?p
WHERE {
  ?s ?p ?o.
}"""
pred_res_list = graph.query(query)

In [53]:
count_total_query = """
SELECT (COUNT(*) AS ?totalCount)
WHERE {
    ?s <:pred> ?o
}
"""

count_total_subj_query = """
SELECT (COUNT(DISTINCT ?s) AS ?totalCount)
WHERE {
    ?s <:pred> ?o
}
"""

func_dict = dict()
for pred_res in tqdm(pred_res_list):
    query_pred = pred_res[0]

    try:
        replaced_count_total_query = count_total_query.replace(':pred', str(query_pred))
        count_res = graph.query(replaced_count_total_query)
        total_count = 0
        for res in count_res:
            total_count = res.totalCount

        replaced_count_total_subj_query = count_total_subj_query.replace(':pred', str(query_pred))
        count_res = graph.query(replaced_count_total_subj_query)
        total_obj = 0
        for res in count_res:
            total_obj = res.totalCount
        
        func_dict[str(query_pred)] = float(total_obj)/float(total_count)
    except Exception as e:
        print(e)
        print("exception raised: " + str(query_pred))

100%|██████████████████████████████████████████| 545/545 [00:23<00:00, 22.86it/s]


In [54]:
func_list = list()
for key, value in func_dict.items():
    func_list.append({
        'predicate': key,
        'functionality': value
    })

In [55]:
func_df = pd.DataFrame(func_list)

In [56]:
func_df

,predicate,functionality
0,http://dbpedia.org/ontology/guest,0.707391
1,http://www.wikidata.org/entity/P345,0.999313
2,http://www.wikidata.org/entity/P161,0.599616
3,http://www.wikidata.org/entity/P175,0.994080
4,http://www.wikidata.org/entity/P577,0.936466
...,...,...
540,http://www.wikidata.org/entity/P38,1.000000
541,http://dbpedia.org/ontology/governmentCountry,1.000000
542,http://dbpedia.org/ontology/feastDay,1.000000
543,http://dbpedia.org/ontology/openingYear,1.000000


In [57]:
func_df[func_df['functionality'] > 0.5]

,predicate,functionality
0,http://dbpedia.org/ontology/guest,0.707391
1,http://www.wikidata.org/entity/P345,0.999313
2,http://www.wikidata.org/entity/P161,0.599616
3,http://www.wikidata.org/entity/P175,0.994080
4,http://www.wikidata.org/entity/P577,0.936466
...,...,...
540,http://www.wikidata.org/entity/P38,1.000000
541,http://dbpedia.org/ontology/governmentCountry,1.000000
542,http://dbpedia.org/ontology/feastDay,1.000000
543,http://dbpedia.org/ontology/openingYear,1.000000


In [58]:
inversability_df = pd.merge(func_df, inv_func_df, on='predicate')
inversability_df['inversability'] = inversability_df['inverse_functionality'] * inversability_df['functionality']


In [59]:
inversability_df

,predicate,functionality,inverse_functionality,inversability
0,http://dbpedia.org/ontology/guest,0.707391,0.551282,0.389972
1,http://www.wikidata.org/entity/P345,0.999313,0.996910,0.996226
2,http://www.wikidata.org/entity/P161,0.599616,0.338014,0.202679
3,http://www.wikidata.org/entity/P175,0.994080,0.195203,0.194048
4,http://www.wikidata.org/entity/P577,0.936466,0.607109,0.568537
...,...,...,...,...
540,http://www.wikidata.org/entity/P38,1.000000,1.000000,1.000000
541,http://dbpedia.org/ontology/governmentCountry,1.000000,1.000000,1.000000
542,http://dbpedia.org/ontology/feastDay,1.000000,1.000000,1.000000
543,http://dbpedia.org/ontology/openingYear,1.000000,1.000000,1.000000


In [60]:
inversability_df[inversability_df['inverse_functionality'] > 0.5]

,predicate,functionality,inverse_functionality,inversability
0,http://dbpedia.org/ontology/guest,0.707391,0.551282,0.389972
1,http://www.wikidata.org/entity/P345,0.999313,0.996910,0.996226
4,http://www.wikidata.org/entity/P577,0.936466,0.607109,0.568537
7,http://www.w3.org/2000/01/rdf-schema#label,1.000000,0.961267,0.961267
9,http://www.wikidata.org/entity/P569,0.750246,0.577504,0.433270
...,...,...,...,...
540,http://www.wikidata.org/entity/P38,1.000000,1.000000,1.000000
541,http://dbpedia.org/ontology/governmentCountry,1.000000,1.000000,1.000000
542,http://dbpedia.org/ontology/feastDay,1.000000,1.000000,1.000000
543,http://dbpedia.org/ontology/openingYear,1.000000,1.000000,1.000000


In [61]:
inversability_df.to_csv('D:/FTM/Full-Triple-Matcher/dataset/entity-matching/DW-NB/kg1_inversability.csv')